# Exercise 2: Exploring the earthquake data

Uses `parsed.csv`, the USGS earthquake data. The `parsed_place` column holds the cleaned-up country or US state for each earthquake.

In [1]:
import pandas as pd

# parsed.csv needs to be in the same folder as this notebook. In Colab, upload it with
# the Files pane on the left, or mount Google Drive and use the file's full path.
df = pd.read_csv('parsed.csv')
df.shape

(9332, 27)

### a. 95th percentile of earthquake magnitude in Japan, using magType `mb`

In [2]:
# Keep only the earthquakes in Japan measured with the mb magnitude type,
# then take the 95th percentile (the 0.95 quantile) of their magnitudes
japan_mb = df.query("parsed_place == 'Japan' and magType == 'mb'")
print(f'95th percentile of mb magnitudes in Japan: {japan_mb.mag.quantile(0.95)} '
      f'(based on {len(japan_mb)} earthquakes)')

95th percentile of mb magnitudes in Japan: 4.9 (based on 50 earthquakes)


### b. Percentage of earthquakes in Indonesia that were coupled with tsunamis

In [3]:
# tsunami is 1 when an earthquake was coupled with a tsunami and 0 when it wasn't,
# so the mean of the column is the share of earthquakes that had a tsunami
indonesia = df.query("parsed_place == 'Indonesia'")
pct_tsunami = indonesia.tsunami.mean() * 100
print(f'{pct_tsunami:.2f}% of earthquakes in Indonesia were coupled with tsunamis '
      f'({indonesia.tsunami.sum()} of {len(indonesia)})')

23.13% of earthquakes in Indonesia were coupled with tsunamis (34 of 147)


### c. Summary statistics for earthquakes in Nevada

In [4]:
# describe() gives the count, mean, standard deviation, minimum, quartiles and
# maximum of every numeric column
df.query("parsed_place == 'Nevada'").describe()

,cdi,dmin,felt,gap,mag,mmi,nst,rms,sig,time,tsunami,tz,updated
count,15.000000,681.000000,15.000000,681.000000,681.000000,1.00,681.000000,681.000000,681.000000,6.810000e+02,681.0,681.0,6.810000e+02
mean,2.440000,0.166199,2.400000,153.668120,0.500073,2.84,12.618209,0.151986,10.970631,1.538314e+12,0.0,-480.0,1.538402e+12
std,0.501142,0.166228,4.626013,68.735302,0.696710,NaN,9.866963,0.084662,19.607150,5.965637e+08,0.0,0.0,6.010951e+08
min,2.000000,0.001000,1.000000,29.140000,-0.500000,2.84,3.000000,0.000500,0.000000,1.537247e+12,0.0,-480.0,1.537307e+12
25%,2.000000,0.053000,1.000000,97.380000,-0.100000,2.84,6.000000,0.106900,0.000000,1.537854e+12,0.0,-480.0,1.537928e+12
50%,2.200000,0.112000,1.000000,149.140000,0.400000,2.84,10.000000,0.146300,2.000000,1.538280e+12,0.0,-480.0,1.538428e+12
75%,2.900000,0.233000,1.000000,199.720000,0.900000,2.84,16.000000,0.187100,12.000000,1.538821e+12,0.0,-480.0,1.538878e+12
max,3.300000,1.414000,19.000000,355.910000,2.900000,2.84,61.000000,0.863400,129.000000,1.539461e+12,0.0,-480.0,1.539483e+12


### d. Flag earthquakes in Ring of Fire countries and US states

In [5]:
# In a regular expression '^' means "starts with", so '^Mexico' matches Mexico but not
# New Mexico. 'Antarctic' matches any place with Antarctic in its name.
ring_of_fire_places = [
    'Bolivia', 'Chile', 'Ecuador', 'Peru', 'Costa Rica', 'Guatemala', '^Mexico',
    'Japan', 'Philippines', 'Indonesia', 'New Zealand', 'Antarctic', 'Canada',
    'Fiji', 'Alaska', 'Washington', 'California', 'Russia', 'Taiwan', 'Tonga',
    'Kermadec Islands',
]

# '|' means "or", so the new column is True when parsed_place contains any of the names
df['ring_of_fire'] = df.parsed_place.str.contains('|'.join(ring_of_fire_places))

# Check which places were flagged (New Mexico should not be one of them)
print(', '.join(sorted(df.loc[df.ring_of_fire, 'parsed_place'].unique())))

Alaska, Bolivia, California, Canada, Chile, Costa Rica, Ecuador, Fiji, Guatemala, Indonesia, Japan, Kermadec Islands, Mexico, New Zealand, Pacific-Antarctic Ridge, Peru, Philippines, Russia, Taiwan, Tonga, Washington, Western Indian-Antarctic Ridge


The data has no place named Antarctica, so the `Antarctic` search picks up two ocean ridges: the Pacific-Antarctic Ridge and the Western Indian-Antarctic Ridge. The counts below follow the list in the assignment, which leaves out some other Pacific-rim places that appear in the data, such as Papua New Guinea, Vanuatu, the Solomon Islands, the Kuril Islands and Oregon.

### e. Number of earthquakes in Ring of Fire locations and outside them

In [6]:
# True = in a Ring of Fire location, False = outside
df.ring_of_fire.value_counts()

ring_of_fire
True     7188
False    2144
Name: count, dtype: int64

### f. Tsunami count along the Ring of Fire

In [7]:
# Adding up the 0/1 tsunami column for the Ring of Fire rows counts their tsunamis
ring_tsunamis = df.loc[df.ring_of_fire, 'tsunami'].sum()
print(f'{ring_tsunamis} tsunamis along the Ring of Fire '
      f'(out of {df.tsunami.sum()} in the whole data set)')

45 tsunamis along the Ring of Fire (out of 61 in the whole data set)
